# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedant08mehta/Flyrank-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/vedant08mehta/Flyrank-assignment1.git
%cd /content/Flyrank-assignment1

Cloning into 'Flyrank-assignment1'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 151 (delta 59), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 1.89 MiB | 23.03 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Flyrank-assignment1


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will prioritize pages that show weak recent search performance while still having meaningful search exposure. The rule will give higher scores to pages with lower CTR, weaker average position, and a decline in recent impressions compared with the previous period. Each page will also receive reason codes explaining why it was prioritized, so the ranking remains interpretable.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

df["impression_change"] = (
    df["impressions_last_30d"]
    - df["impressions_prev_30d"]
) / df["impressions_prev_30d"].replace(0, np.nan)

df["action_score"] = (
    (1 - df["ctr"].rank(pct=True))
    + df["avg_position"].rank(pct=True)
    + (1 - df["impression_change"].rank(pct=True))
)

df["reason_code"] = np.select(
    [
        df["ctr"] <= df["ctr"].median(),
        df["avg_position"] >= df["avg_position"].median(),
        df["impression_change"] < 0
    ],
    [
        "LOW_CTR",
        "WEAK_POSITION",
        "IMPRESSION_DECLINE"
    ],
    default="GENERAL_REVIEW"
)

print(f"Rows scored: {len(df):,}")
print("\nReason codes:")
print(df["reason_code"].value_counts())

Rows scored: 30,000

Reason codes:
reason_code
LOW_CTR               15224
WEAK_POSITION          6608
IMPRESSION_DECLINE     5987
GENERAL_REVIEW         2181
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I will rank every content page by the hand-built action score, with higher scores representing greater review priority. The resulting queue will be saved as outputs/baseline_action_score.csv so that the ranking can be inspected separately from the model.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

baseline_queue = (
    df[
        [
            "content_id",
            "action_score",
            "reason_code",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "impression_change"
        ]
    ]
    .sort_values(
        "action_score",
        ascending=False
    )
    .reset_index(drop=True)
)

baseline_queue.to_csv(
    output_dir / "baseline_action_score.csv",
    index=False
)

print(
    f"Saved {len(baseline_queue):,} ranked pages."
)

display(
    baseline_queue.head(10)
)

Saved 30,000 ranked pages.


,content_id,action_score,reason_code,impressions_90d,clicks_90d,ctr,avg_position,impression_change
0,content_13bbd72aea33,2.753288,LOW_CTR,3,0,0.0,118.0,-1.0
1,content_638236e8066e,2.752921,LOW_CTR,1,0,0.0,98.0,-1.0
2,content_f616ca0ec5ea,2.752588,LOW_CTR,1,0,0.0,94.0,-1.0
3,content_86748254b6bf,2.752455,LOW_CTR,1,0,0.0,90.0,-1.0
4,content_4ed2bf493735,2.751005,LOW_CTR,1,0,0.0,83.0,-1.0
5,content_584e85b1ef21,2.750655,LOW_CTR,5,0,0.0,81.8,-1.0
6,content_a1c4525abad0,2.750455,LOW_CTR,39,0,0.0,81.3,-1.0
7,content_930c7a86e456,2.750138,LOW_CTR,108,0,0.0,80.2,-1.0
8,content_51bbaf502f53,2.749621,LOW_CTR,13,0,0.0,78.8,-1.0
9,content_f5ede7978b69,2.749588,LOW_CTR,36,0,0.0,78.7,-1.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages represent the pages that the hand-written rule considers highest priority for review. For each page, I will record the recommended action, the reason code, a confidence note, and what evidence could make the recommendation wrong. The rule is intended for prioritization, not as an automatic decision.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline_queue.head(20).copy()

top20["action"] = "Review page"

top20["confidence_note"] = np.where(
    top20["action_score"] >= top20["action_score"].median(),
    "Higher-priority rule match",
    "Lower-confidence rule match"
)

top20["what_could_make_it_wrong"] = (
    "Low exposure, unusual content type, "
    "or another factor not captured by the rule"
)

display(
    top20[
        [
            "content_id",
            "action",
            "reason_code",
            "action_score",
            "confidence_note",
            "what_could_make_it_wrong"
        ]
    ]
)

,content_id,action,reason_code,action_score,confidence_note,what_could_make_it_wrong
0,content_13bbd72aea33,Review page,LOW_CTR,2.753288,Higher-priority rule match,"Low exposure, unusual content type, or another..."
1,content_638236e8066e,Review page,LOW_CTR,2.752921,Higher-priority rule match,"Low exposure, unusual content type, or another..."
2,content_f616ca0ec5ea,Review page,LOW_CTR,2.752588,Higher-priority rule match,"Low exposure, unusual content type, or another..."
3,content_86748254b6bf,Review page,LOW_CTR,2.752455,Higher-priority rule match,"Low exposure, unusual content type, or another..."
4,content_4ed2bf493735,Review page,LOW_CTR,2.751005,Higher-priority rule match,"Low exposure, unusual content type, or another..."
5,content_584e85b1ef21,Review page,LOW_CTR,2.750655,Higher-priority rule match,"Low exposure, unusual content type, or another..."
6,content_a1c4525abad0,Review page,LOW_CTR,2.750455,Higher-priority rule match,"Low exposure, unusual content type, or another..."
7,content_930c7a86e456,Review page,LOW_CTR,2.750138,Higher-priority rule match,"Low exposure, unusual content type, or another..."
8,content_51bbaf502f53,Review page,LOW_CTR,2.749621,Higher-priority rule match,"Low exposure, unusual content type, or another..."
9,content_f5ede7978b69,Review page,LOW_CTR,2.749588,Higher-priority rule match,"Low exposure, unusual content type, or another..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are pages where the score may be driven by only one signal or where low exposure makes the recommendation less reliable. I will also explicitly check that target-derived trend fields and other outcome information are not being used in the baseline score. trend_direction, trend_pct, and is_declining_label are excluded because they directly describe or define the outcome and would make the baseline leaky.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

score_inputs = [
    "ctr",
    "avg_position",
    "impression_change"
]

print("=== Leakage check ===")

for col in excluded:
    print(
        f"{col}: "
        f"{'USED' if col in score_inputs else 'NOT USED'}"
    )

assert not any(
    col in score_inputs
    for col in excluded
)

print(
    "\nLeakage check passed: "
    "no target-derived fields are used."
)


print("\n=== Weakest top-20 picks ===")

display(
    top20.tail(5)[
        [
            "content_id",
            "reason_code",
            "action_score",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ]
)

=== Leakage check ===
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED

Leakage check passed: no target-derived fields are used.

=== Weakest top-20 picks ===


,content_id,reason_code,action_score,impressions_90d,ctr,avg_position
15,content_b7e5b7cf98e3,LOW_CTR,2.748588,233,0.0,76.6
16,content_7764f406228e,LOW_CTR,2.748205,1,0.0,76.0
17,content_9a9fe9053852,LOW_CTR,2.748205,56,0.0,76.0
18,content_16064dfe2d43,LOW_CTR,2.748055,25,0.0,75.8
19,content_cefdb6f4c6ae,LOW_CTR,2.747255,131,0.0,74.4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.